# 🎨 SD Image Generator
Stable Diffusion XL · ipywidgets UI · YouTube-ready sizes · Face mode

In [ ]:
import subprocess, sys, os

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"

def install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

PACKAGES = [
    "diffusers",
    "accelerate",
    "transformers",
    "huggingface_hub",
    "ipywidgets",
    "insightface",
    "onnxruntime-gpu",
    "opencv-python-headless",
]
for pkg in PACKAGES:
    install(pkg)

import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import random, io, base64
from PIL import Image

print(f"✅ Ready | PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
STYLE_MODELS = {
    "Realism":       "SG161222/RealVisXL_V4.0",
    "Anime":         "cagliostrolab/animagine-xl-3.1",
    "Comics":        "Lykon/dreamshaper-xl-1-0",
    "Illustration":  "playgroundai/playground-v2.5-1024px-aesthetic",
    "Lineart":       "stabilityai/stable-diffusion-xl-base-1.0",
}

SIZES = {
    "YouTube Video (1920×1080)":  (1920, 1080),
    "YouTube Shorts (1080×1920)": (1080, 1920),
}

from diffusers import DiffusionPipeline, StableDiffusionXLPipeline
from huggingface_hub import file_exists, list_repo_files, hf_hub_url

def load_pipeline(model_id: str):
    try:
        has_diffusers_format = file_exists(model_id, "model_index.json")
    except Exception:
        has_diffusers_format = False

    if has_diffusers_format:
        try:
            pipe = DiffusionPipeline.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                use_safetensors=True,
                variant="fp16",
            )
        except Exception:
            pipe = DiffusionPipeline.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                use_safetensors=True,
            )
    else:
        files = list(list_repo_files(model_id))
        sf = next((f for f in files if f.endswith(".safetensors") and "/" not in f), None)
        if sf is None:
            raise ValueError(f"No root-level .safetensors found in {model_id}")
        url = hf_hub_url(model_id, filename=sf)
        pipe = StableDiffusionXLPipeline.from_single_file(
            url,
            torch_dtype=torch.float16,
            use_safetensors=True,
        )

    pipe.enable_model_cpu_offload()
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass
    return pipe

pipe = None

style_radio = widgets.RadioButtons(
    options=list(STYLE_MODELS.keys()),
    description="Style:",
    layout=widgets.Layout(width="400px"),
)
load_btn = widgets.Button(
    description="⚙️ Load model",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)
model_status = widgets.Label(value="No model loaded")

def on_load(btn):
    global pipe
    model_status.value = f"⏳ Loading {style_radio.value}..."
    load_btn.disabled = True
    try:
        pipe = load_pipeline(STYLE_MODELS[style_radio.value])
        model_status.value = f"✅ {style_radio.value} ready"
    except Exception as e:
        model_status.value = f"❌ {e}"
    finally:
        load_btn.disabled = False

load_btn.on_click(on_load)
display(widgets.VBox([
    widgets.Label("🎨 Select style and load model:"),
    style_radio,
    widgets.HBox([load_btn, model_status]),
]))

In [ ]:
def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def show_gallery(images, seeds_used):
    cards = []
    for i, (img, seed) in enumerate(zip(images, seeds_used)):
        b64 = image_to_b64(img)
        dl = widgets.HTML(
            f'<a download="img_{i+1}_seed{seed}.png" href="data:image/png;base64,{b64}">'
            f'<button>⬇ Download</button></a>'
        )
        thumb = widgets.Image(value=base64.b64decode(b64), format="png", width=320, height=180)
        cards.append(widgets.VBox([thumb, dl]))
    rows = [widgets.HBox(cards[i:i+4]) for i in range(0, len(cards), 4)]
    display(widgets.VBox(rows))

def generate_images(pipe, prompt, negative_prompt, seed, count, width, height, guidance):
    generator = torch.Generator(device="cuda").manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_images_per_prompt=count,
        width=width, height=height,
        generator=generator,
        num_inference_steps=30,
        guidance_scale=guidance,
    ).images

def load_face_pipeline(pipe, face_bytes):
    if not getattr(pipe, "_ip_adapter_loaded", False):
        face_img = Image.open(io.BytesIO(face_bytes)).convert("RGB")
        pipe.load_ip_adapter(
            "h94/IP-Adapter",
            subfolder="sdxl_models",
            weight_name="ip-adapter-plus-face_sdxl_vit-h.safetensors",
        )
        pipe.set_ip_adapter_scale(0.7)
        pipe._ip_adapter_loaded = True
    else:
        face_img = Image.open(io.BytesIO(face_bytes)).convert("RGB")
    return pipe, face_img

def generate_with_face(pipe, face_img, prompt, negative_prompt, seed, count, width, height, guidance):
    generator = torch.Generator(device="cuda").manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        ip_adapter_image=face_img,
        num_images_per_prompt=count,
        width=width, height=height,
        generator=generator,
        num_inference_steps=30,
        guidance_scale=guidance,
    ).images

prompt_ta = widgets.Textarea(
    placeholder="Describe what you want to generate...",
    description="Prompt:",
    layout=widgets.Layout(width="600px", height="80px"),
)
neg_prompt_ta = widgets.Textarea(
    value="deformed, ugly, blurry, low quality, worst quality",
    description="Negative:",
    layout=widgets.Layout(width="600px", height="60px"),
)
count_slider = widgets.IntSlider(
    value=2, min=1, max=8, step=1,
    description="Count:",
    layout=widgets.Layout(width="400px"),
)
guidance_slider = widgets.FloatSlider(
    value=7.5, min=1.0, max=15.0, step=0.5,
    description="Guidance:",
    layout=widgets.Layout(width="400px"),
)
seed_input = widgets.IntText(
    value=42,
    description="Seed:",
    layout=widgets.Layout(width="200px"),
)
random_seed_btn = widgets.Button(
    description="🎲 Random seed",
    layout=widgets.Layout(width="160px"),
)
seed_mode_radio = widgets.RadioButtons(
    options=["Fixed", "Random each time"],
    value="Fixed",
    description="Seed mode:",
)
size_radio = widgets.RadioButtons(
    options=list(SIZES.keys()),
    description="Size:",
)
face_upload = widgets.FileUpload(
    accept="image/*",
    description="Face (opt.):",
    layout=widgets.Layout(width="300px"),
)
generate_btn = widgets.Button(
    description="🚀 Generate",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Progress:",
    layout=widgets.Layout(width="400px"),
)
status_label = widgets.Label(value="Ready")
gallery_output = widgets.Output()

def get_seed():
    if seed_mode_radio.value == "Random each time":
        return random.randint(0, 2**32 - 1)
    return seed_input.value

def on_random_seed(btn):
    seed_input.value = random.randint(0, 2**32 - 1)
random_seed_btn.on_click(on_random_seed)

def on_generate(btn):
    if pipe is None:
        status_label.value = "❌ Load a model first (run the cell above)"
        return
    generate_btn.disabled = True
    progress_bar.value = 0
    status_label.value = "⏳ Generating..."
    try:
        seed = get_seed()
        seed_input.value = seed
        width, height = SIZES[size_radio.value]
        guidance = guidance_slider.value
        face_bytes = None
        if face_upload.value:
            face_bytes = face_upload.value[0]["content"]
        progress_bar.value = 30
        if face_bytes:
            updated_pipe, face_img = load_face_pipeline(pipe, face_bytes)
            images = generate_with_face(
                updated_pipe, face_img,
                prompt_ta.value, neg_prompt_ta.value,
                seed, count_slider.value, width, height, guidance,
            )
        else:
            if getattr(pipe, "_ip_adapter_loaded", False):
                pipe.unload_ip_adapter()
                pipe._ip_adapter_loaded = False
            images = generate_images(
                pipe,
                prompt_ta.value, neg_prompt_ta.value,
                seed, count_slider.value, width, height, guidance,
            )
        progress_bar.value = 90
        with gallery_output:
            clear_output(wait=True)
            show_gallery(images, [seed] * len(images))
        progress_bar.value = 100
        status_label.value = f"✅ Done — {len(images)} image(s) | seed: {seed}"
    except Exception as e:
        status_label.value = f"❌ Error: {e}"
    finally:
        generate_btn.disabled = False

generate_btn.on_click(on_generate)

display(widgets.VBox([
    prompt_ta,
    neg_prompt_ta,
    widgets.HBox([count_slider, guidance_slider]),
    widgets.HBox([seed_input, random_seed_btn]),
    seed_mode_radio,
    size_radio,
    face_upload,
    generate_btn,
    widgets.HBox([progress_bar, status_label]),
]))
display(gallery_output)
